# 🎬 Estúdio — rodar no Google Colab (1x T4)

O Kaggle (2x T4) é mais rápido: use `kaggle/estudio.ipynb` se puder.
Aqui, ative **Ambiente de execução → Alterar tipo → GPU T4** e rode as células em ordem.
Na T4 única, o codificador de texto é carregado em 8 bits para caber.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
RAMO = "estudio-video"
!rm -rf Jarvis && git clone -q -b {RAMO} --depth 1 https://github.com/gutodrac-spec/Jarvis.git
%cd Jarvis/estudio
!pip install -q -r requirements.txt

In [ ]:
!ESTUDIO_SHARE=1 ESTUDIO_PRECARREGAR=1 python app.py

## Publicar como Hugging Face Space (opcional)
Precisa do segredo `HF_TOKEN` (token *write*). O Space grátis só tem CPU, que é lento demais para vídeo:
escolha uma GPU nas configurações do Space ou use **ZeroGPU** (conta PRO). Pare a célula que roda o app antes.

In [ ]:
from google.colab import userdata
HF_TOKEN = userdata.get("HF_TOKEN")
from huggingface_hub import HfApi

NOME_DO_SPACE = "estudio"  # vira https://huggingface.co/spaces/<seu-usuario>/estudio

api = HfApi(token=HF_TOKEN)
repo = f"{api.whoami()['name']}/{NOME_DO_SPACE}"
api.create_repo(repo, repo_type="space", space_sdk="gradio", exist_ok=True)
with open("/tmp/README.md", "w") as f:
    f.write("---\ntitle: Estúdio\nemoji: 🎬\nsdk: gradio\napp_file: app.py\npinned: false\n---\n\n"
            "Texto ou imagem para vídeo com presets de câmera.\n")
api.upload_folder(repo_id=repo, repo_type="space", folder_path=".",
                  allow_patterns=["app.py", "presets.py", "requirements.txt"])
api.upload_file(repo_id=repo, repo_type="space", path_or_fileobj="/tmp/README.md", path_in_repo="README.md")
print("Space:", f"https://huggingface.co/spaces/{repo}")